# Graph 5: Netflix Catalog Treemap

In [ ]:
from IPython.display import HTML, display

html_content = """<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8">
<style>body{margin:0;padding:0;background:#f5f0eb;}</style>
</head>
<body>
<div id="treemap"></div>
<script src="https://d3js.org/d3.v7.min.js"></script>
<script>
// Graph 5: Netflix Catalog Treemap
// Loads data from graph5_treemap_data.csv (generated by graph5_clean_data.py)
// Three levels: Netflix catalog -> content type -> genre

const movieColors = [
  "#1a5f7a","#2d7d9a","#3a9cbf","#5bb8d4","#7fcce0",
  "#a3dded","#c4ecf5","#1d6b55","#2d9070","#4ab38e"
];
const tvColors = [
  "#8b1a1a","#b52424","#d43d3d","#e86a6a","#f29090",
  "#f5aaaa","#e8452a","#c43520","#f07050","#f5a090"
];

const csvString = `type,genre,value
Movie,International Movies,2548
Movie,Dramas,2398
Movie,Comedies,1636
Movie,Action & Adventure,848
Movie,Independent Movies,751
Movie,Romantic Movies,604
Movie,Children & Family Movies,583
Movie,Thrillers,575
Movie,Documentaries,418
Movie,Horror Movies,352
TV Show,International TV Shows,107
TV Show,TV Dramas,61
TV Show,Crime TV Shows,45
TV Show,TV Comedies,39
TV Show,Romantic TV Shows,28
TV Show,British TV Shows,21
TV Show,Docuseries,17
TV Show,TV Action & Adventure,16
TV Show,Kids' TV,14
TV Show,Stand-Up Comedy & Talk Shows,12`;
d3.csvParse(csvString, d3.autoType);
Promise.resolve(d3.csvParse(csvString, d3.autoType)).then(function(data) {

  // Parse value to number
  data.forEach(d => { d.value = +d.value; });

  // Build three-level hierarchy from flat CSV
  const typeMap = {};
  data.forEach(d => {
    if (!typeMap[d.type]) typeMap[d.type] = [];
    typeMap[d.type].push({ name: d.genre, value: d.value });
  });

  const treemapData = {
    name: "Netflix",
    children: Object.entries(typeMap).map(([type, children]) => ({
      name: type,
      children: children
    }))
  };

  // -- Layout ------------------------------------------------------------------
  const W = 900, H = 540;
  const margin = { top: 50, right: 10, bottom: 10, left: 10 };
  const innerW = W - margin.left - margin.right;
  const innerH = H - margin.top - margin.bottom;

  const svg = d3.select("#treemap")
    .append("svg")
    .attr("width", W).attr("height", H)
    .style("font-family", "Arial, sans-serif")
    .style("background", "#f5f0eb");

  // Title
  svg.append("text")
    .attr("x", W / 2).attr("y", 28)
    .style("text-anchor", "middle").style("font-size", "14px")
    .style("font-weight", "bold").style("fill", "#222")
    .text("Netflix Catalog: Content Type → Genre");

  svg.append("text")
    .attr("x", W / 2).attr("y", 44)
    .style("text-anchor", "middle").style("font-size", "10px")
    .style("fill", "#888")
    .text("Size = number of titles  ·  Hover for details");

  const g = svg.append("g")
    .attr("transform", `translate(${margin.left},${margin.top})`);

  // -- Hierarchy ---------------------------------------------------------------
  const root = d3.hierarchy(treemapData)
    .sum(d => d.value)
    .sort((a, b) => b.value - a.value);

  d3.treemap()
    .size([innerW, innerH])
    .paddingOuter(6)
    .paddingInner(2)
    .paddingTop(22)
    .round(true)(root);

  // -- Tooltip -----------------------------------------------------------------
  const tooltip = d3.select("body").append("div")
    .style("position", "absolute")
    .style("background", "rgba(20,20,20,0.92)")
    .style("color", "#fff")
    .style("padding", "10px 14px")
    .style("border-radius", "6px")
    .style("font-size", "12px")
    .style("pointer-events", "none")
    .style("visibility", "hidden")
    .style("line-height", "1.8");

  // -- Draw nodes --------------------------------------------------------------
  const nodes = g.selectAll("g")
    .data(root.descendants().filter(d => d.depth > 0))
    .enter().append("g")
    .attr("transform", d => `translate(${d.x0},${d.y0})`);

  nodes.append("rect")
    .attr("width",  d => Math.max(0, d.x1 - d.x0))
    .attr("height", d => Math.max(0, d.y1 - d.y0))
    .attr("rx", 3)
    .attr("fill", d => {
      if (d.depth === 1) {
        return d.data.name === "Movie" ? "#0d3d52" : "#5c0f0f";
      }
      const isMovie = d.parent.data.name === "Movie";
      const idx = d.parent.children.indexOf(d);
      return isMovie ? movieColors[idx % movieColors.length] : tvColors[idx % tvColors.length];
    })
    .attr("opacity", d => d.depth === 1 ? 1 : 0.88)
    .attr("stroke", "#f5f0eb")
    .attr("stroke-width", d => d.depth === 1 ? 0 : 0.5)
    .style("cursor", "default")
    .on("mouseover", function(event, d) {
      if (d.depth === 2) {
        d3.select(this).attr("opacity", 1);
        const pct      = ((d.value / root.value) * 100).toFixed(1);
        const parentPct = ((d.value / d.parent.value) * 100).toFixed(1);
        tooltip.style("visibility", "visible")
          .html(
            `<strong>${d.data.name}</strong><br>` +
            `Type: ${d.parent.data.name}s<br>` +
            `Titles: <strong>${d.value.toLocaleString()}</strong><br>` +
            `${parentPct}% of ${d.parent.data.name}s<br>` +
            `${pct}% of total catalog`
          );
      }
    })
    .on("mousemove", function(event) {
      tooltip
        .style("top",  (event.pageY - 10) + "px")
        .style("left", (event.pageX + 14) + "px");
    })
    .on("mouseout", function(event, d) {
      d3.select(this).attr("opacity", d.depth === 1 ? 1 : 0.88);
      tooltip.style("visibility", "hidden");
    });

  // -- Labels: depth 1 (parent group header) -----------------------------------
  nodes.filter(d => d.depth === 1)
    .append("text")
    .attr("x", 6).attr("y", 15)
    .style("font-size", "12px").style("font-weight", "bold").style("fill", "#fff")
    .text(d => `${d.data.name}s  (${d.value.toLocaleString()} titles)`);

  // -- Labels: depth 2 (genre blocks — only if big enough) --------------------
  nodes.filter(d => d.depth === 2)
    .each(function(d) {
      const bw = d.x1 - d.x0;
      const bh = d.y1 - d.y0;
      if (bw < 48 || bh < 20) return;

      const cell  = d3.select(this);
      const lineH = 13;
      const maxLines = Math.floor((bh - 18) / lineH);
      const words = d.data.name.split(" ");

      const lines = [];
      let line = "";
      for (const word of words) {
        if ((line + " " + word).trim().length > Math.floor(bw / 7)) {
          if (line) lines.push(line);
          line = word;
        } else {
          line = (line + " " + word).trim();
        }
      }
      if (line) lines.push(line);

      const displayLines = lines.slice(0, maxLines);
      displayLines.forEach((l, i) => {
        cell.append("text")
          .attr("x", 4).attr("y", 14 + i * lineH)
          .style("font-size", "10px").style("fill", "#fff")
          .style("pointer-events", "none")
          .text(l);
      });

      if (bh > 38) {
        cell.append("text")
          .attr("x", 4).attr("y", 14 + displayLines.length * lineH + 2)
          .style("font-size", "9px").style("fill", "rgba(255,255,255,0.7)")
          .style("pointer-events", "none")
          .text(d.value.toLocaleString());
      }
    });

}).catch(err => {
  d3.select("#treemap").append("p")
    .style("color","red").style("padding","20px")
    .text("Could not load graph5_treemap_data.csv — make sure it is in the same folder.");
});

</script>
</body>
</html>"""

display(HTML(html_content))